# LI-Small vs 无标签交易数据：相似性检验（Fusion）

> 目标：在无欺诈标签场景下，构建"可迁移性证据链"。本Notebook实现 Fusion.md 中的 **相似性检验**：

- **表格层（交易级）**：
  - 金额分布对比（LI-Small `amount` vs 无标签 `payment_amount`）
  - 交易方向性对比（账户活跃度、交易次数分布等）
- **图结构层（最关键）**：对齐账户交易图的结构统计（度分布、连通分量、重复边、reciprocity）

> 注：无标签数据字段来自 `Fraud Graph字段含义.md`；LI-Small 数据来自 `Graph/Fusion Model/raw_data/LI-Small_Trans.csv`（洗钱模拟数据集）。

In [ ]:
# ============ 配置区（按需修改） ============
from pathlib import Path

# LI-Small 数据（洗钱模拟交易数据集）
LI_TRANS_PATH = Path(r"Graph/Fusion Model/raw_data/LI-Small_Trans.csv")

# 无标签数据：请改成你的真实交易CSV路径
UNLABELED_PATH = Path(r"Graph/graph_main/raw_data/xxx.csv")

# LI-Small 字段名映射（基于 LI-Small_Patterns.txt 格式推断）
# 格式：timestamp, src_account, src_id, dst_account, dst_id, amount, currency1, amount2, currency2, channel, label
LI_COLS = {
    "timestamp": 0,  # 列索引
    "src": 1,        # 源账户
    "dst": 3,        # 目标账户
    "amount": 5,     # 金额
    "currency": 6,   # 币种
    "channel": 9,    # 渠道
}

# 无标签数据字段名
UNLABELED_COLS = {
    "amount": "payment_amount",
    "src": "debit_account_masked",
    "dst": "bene_account_masked",
    "currency": "payment_currency",
    "channel": "payment_channel",
}

# 采样与统计参数
RANDOM_SEED = 42
LI_SAMPLE_SIZE = 500000  # LI-Small 采样行数（0表示全量，但文件很大建议采样）
UNLABELED_SAMPLE_SIZE = 0  # 无标签数据采样（0表示全量）
TOP_Q = [0.50, 0.90, 0.95, 0.99, 0.999]
HIST_BINS = 200

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.stats import wasserstein_distance

sns.set_theme(style="whitegrid")
np.random.seed(RANDOM_SEED)
plt.rcParams["figure.figsize"] = (10, 4)


In [ ]:
def _safe_read_csv(path: Path, nrows: int = 0) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"找不到文件: {path}")
    return pd.read_csv(path, nrows=None if nrows == 0 else nrows)

def _read_li_small(path: Path, sample_size: int = 0) -> pd.DataFrame:
    """读取 LI-Small_Trans.csv（可能无表头，按固定格式解析）"""
    if not path.exists():
        raise FileNotFoundError(f"找不到文件: {path}")
    
    # 先读取5行判断是否有表头
    preview = pd.read_csv(path, nrows=5, header=None)
    has_header = False
    
    # 如果第一行包含非数字的时间格式字符串，可能是表头或数据
    try:
        pd.to_datetime(preview.iloc[0, 0])
        has_header = False  # 第一行可解析为时间，说明是数据
    except:
        has_header = True
    
    # 读取数据
    if sample_size > 0:
        df = pd.read_csv(path, header=0 if has_header else None, nrows=sample_size)
    else:
        df = pd.read_csv(path, header=0 if has_header else None)
    
    # 如果无表头，按索引访问列
    if not has_header:
        df.columns = [f"col_{i}" for i in range(len(df.columns))]
    
    return df

def _to_numeric_series(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce")

def _ensure_columns(df: pd.DataFrame, cols: list[str], df_name: str):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(f"{df_name} 缺少列: {missing}. 现有列示例: {list(df.columns)[:30]}")

def _qcurve(arr: np.ndarray, qs: list[float]) -> pd.DataFrame:
    arr = arr[~np.isnan(arr)]
    return pd.DataFrame({"q": qs, "value": [np.quantile(arr, q) for q in qs]})

def _tail_share(arr: np.ndarray, q: float) -> float:
    """Top-q_tail(如0.99)占比：大于等于该分位数的总和 / 总和"""
    arr = arr[~np.isnan(arr)]
    if arr.size == 0:
        return np.nan
    thr = np.quantile(arr, q)
    denom = arr.sum()
    if denom == 0:
        return np.nan
    return float(arr[arr >= thr].sum() / denom)

## 1) 读取数据 & 字段对齐

本节会：
- 读取 `LI-Small_Trans.csv`（洗钱模拟数据，可能无表头，按固定列位置解析）
- 读取无标签CSV，并用 `UNLABELED_COLS` 映射找到金额/账户列
- 做基础清洗：金额转数值、去除 NaN/Inf
- 统一成标准格式：`src`, `dst`, `amount`

In [ ]:
# 读取 LI-Small
print("正在读取 LI-Small 数据...")
li_raw = _read_li_small(LI_TRANS_PATH, sample_size=LI_SAMPLE_SIZE)
print(f"LI-Small 原始 shape: {li_raw.shape}")
print(f"LI-Small 列名: {list(li_raw.columns)[:15]}")

# 根据列索引或列名提取字段
if li_raw.columns[0].startswith("col_"):  # 无表头，按索引
    li = pd.DataFrame({
        "src": li_raw.iloc[:, LI_COLS["src"]].astype(str),
        "dst": li_raw.iloc[:, LI_COLS["dst"]].astype(str),
        "amount": _to_numeric_series(li_raw.iloc[:, LI_COLS["amount"]]),
    })
else:  # 有表头，按列名（需要根据实际情况调整）
    # 这里假设有表头时列名包含 src/dst/amount 关键字
    li = pd.DataFrame({
        "src": li_raw.iloc[:, 1].astype(str),
        "dst": li_raw.iloc[:, 3].astype(str),
        "amount": _to_numeric_series(li_raw.iloc[:, 5]),
    })

# 读取无标签数据
print("\n正在读取无标签数据...")
unlabeled_raw = _safe_read_csv(UNLABELED_PATH, nrows=UNLABELED_SAMPLE_SIZE)
_ensure_columns(unlabeled_raw, list(UNLABELED_COLS.values()), "Unlabeled")

unlabeled = pd.DataFrame({
    "src": unlabeled_raw[UNLABELED_COLS["src"]].astype(str),
    "dst": unlabeled_raw[UNLABELED_COLS["dst"]].astype(str),
    "amount": _to_numeric_series(unlabeled_raw[UNLABELED_COLS["amount"]]),
})

# 清洗：去除 NaN/Inf
li["amount"] = li["amount"].replace([np.inf, -np.inf], np.nan)
unlabeled["amount"] = unlabeled["amount"].replace([np.inf, -np.inf], np.nan)

print("\n=== 数据概览 ===")
print(f"LI-Small shape: {li.shape}")
print(f"Unlabeled shape: {unlabeled.shape}")
print(f"\nLI-Small amount valid: {int(li['amount'].notna().sum())} / {len(li)}")
print(f"Unlabeled amount valid: {int(unlabeled['amount'].notna().sum())} / {len(unlabeled)}")
print(f"\nLI-Small 前5行:\n{li.head()}")
print(f"\nUnlabeled 前5行:\n{unlabeled.head()}")

## 2) 表格层（交易级）：金额分布相似性

对齐字段：
- LI-Small：`amount`（洗钱模拟数据集中的交易金额）
- 无标签：`payment_amount`

输出：
- 原始金额与 `log1p(amount)` 的直方图/核密度对比
- KS 距离、Wasserstein 距离
- 分位数曲线对比（Quantile Curve）
- 尾部占比：Top 1% / Top 0.1% 金额贡献

In [ ]:
# 清洗并转numpy
la = li["amount"].dropna().to_numpy(dtype=float)
ua = unlabeled["amount"].dropna().to_numpy(dtype=float)

# 基础指标
amt_stats = pd.DataFrame({
    "dataset": ["LI-Small", "Unlabeled"],
    "n": [len(la), len(ua)],
    "mean": [la.mean() if len(la) else np.nan, ua.mean() if len(ua) else np.nan],
    "median": [np.median(la) if len(la) else np.nan, np.median(ua) if len(ua) else np.nan],
    "max": [la.max() if len(la) else np.nan, ua.max() if len(ua) else np.nan],
    "min": [la.min() if len(la) else np.nan, ua.min() if len(ua) else np.nan],
})
amt_stats

In [ ]:
# KS / Wasserstein（对原始amount；建议同时看log1p后更稳定）
ks_raw = stats.ks_2samp(la, ua)
wd_raw = wasserstein_distance(la, ua)

ks_log = stats.ks_2samp(np.log1p(la), np.log1p(ua))
wd_log = wasserstein_distance(np.log1p(la), np.log1p(ua))

dist_tbl = pd.DataFrame([
    {"space": "raw", "ks_stat": ks_raw.statistic, "ks_pvalue": ks_raw.pvalue, "wasserstein": wd_raw},
    {"space": "log1p", "ks_stat": ks_log.statistic, "ks_pvalue": ks_log.pvalue, "wasserstein": wd_log},
])
dist_tbl

In [ ]:
# 可视化：log1p金额分布对比
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(np.log1p(la), bins=HIST_BINS, stat="density", kde=True, ax=axes[0], color="#4c78a8", label="LI-Small")
sns.histplot(np.log1p(ua), bins=HIST_BINS, stat="density", kde=True, ax=axes[0], color="#f58518", label="Unlabeled")
axes[0].set_title("log1p(amount) 分布")
axes[0].legend()

# 分位数曲线
lq = _qcurve(la, TOP_Q).rename(columns={"value": "LI-Small"})
uq = _qcurve(ua, TOP_Q).rename(columns={"value": "Unlabeled"})
qdf = lq.merge(uq, on="q", how="outer").sort_values("q")
axes[1].plot(qdf["q"], qdf["LI-Small"], marker="o", label="LI-Small")
axes[1].plot(qdf["q"], qdf["Unlabeled"], marker="o", label="Unlabeled")
axes[1].set_xscale("linear")
axes[1].set_yscale("log")
axes[1].set_title("分位数曲线（y为log刻度）")
axes[1].set_xlabel("quantile")
axes[1].set_ylabel("amount")
axes[1].legend()

plt.tight_layout()
plt.show()

qdf

In [ ]:
# 尾部占比：Top 1% / Top 0.1% 金额贡献
tail_tbl = pd.DataFrame([
    {"dataset": "LI-Small", "top1%_share": _tail_share(la, 0.99), "top0.1%_share": _tail_share(la, 0.999)},
    {"dataset": "Unlabeled", "top1%_share": _tail_share(ua, 0.99), "top0.1%_share": _tail_share(ua, 0.999)},
])
tail_tbl

## 3) 表格层（交易级）：交易方向性对比

对比内容：
- 账户活跃度：唯一 `src` 和 `dst` 数量
- 交易次数分布：每个 `src` 的出账次数、每个 `dst` 的入账次数
- 交易金额聚合：每个 `src` 的总出账金额、每个 `dst` 的总入账金额

这些指标反映了账户网络的"集中度"与"活跃模式"，是图结构对比的前置校验。

In [ ]:
# 计算账户活跃度
def _account_activity(df: pd.DataFrame) -> dict:
    """计算账户级统计：唯一账户数、交易次数分布、金额分布"""
    src_counts = df.groupby("src").size()
    dst_counts = df.groupby("dst").size()
    
    src_amounts = df.groupby("src")["amount"].sum()
    dst_amounts = df.groupby("dst")["amount"].sum()
    
    return {
        "unique_src": int(df["src"].nunique()),
        "unique_dst": int(df["dst"].nunique()),
        "unique_accounts": int(len(set(df["src"]) | set(df["dst"]))),
        "src_txn_count_mean": float(src_counts.mean()),
        "src_txn_count_median": float(src_counts.median()),
        "src_txn_count_max": int(src_counts.max()),
        "dst_txn_count_mean": float(dst_counts.mean()),
        "dst_txn_count_median": float(dst_counts.median()),
        "dst_txn_count_max": int(dst_counts.max()),
        "src_amount_mean": float(src_amounts.mean()),
        "src_amount_median": float(src_amounts.median()),
        "dst_amount_mean": float(dst_amounts.mean()),
        "dst_amount_median": float(dst_amounts.median()),
    }

li_activity = _account_activity(li.dropna(subset=["src", "dst", "amount"]))
u_activity = _account_activity(unlabeled.dropna(subset=["src", "dst", "amount"]))

activity_df = pd.DataFrame([
    {"dataset": "LI-Small", **li_activity},
    {"dataset": "Unlabeled", **u_activity},
])
activity_df

In [ ]:
# 可视化：交易次数分布（出账/入账）
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# LI-Small 出账次数
li_src_counts = li.dropna(subset=["src"]).groupby("src").size().to_numpy()
axes[0, 0].hist(np.log1p(li_src_counts), bins=100, alpha=0.7, color="#4c78a8", edgecolor="black")
axes[0, 0].set_title("LI-Small: 出账次数分布 (log1p)")
axes[0, 0].set_xlabel("log1p(出账次数)")
axes[0, 0].set_ylabel("账户数")

# Unlabeled 出账次数
u_src_counts = unlabeled.dropna(subset=["src"]).groupby("src").size().to_numpy()
axes[0, 1].hist(np.log1p(u_src_counts), bins=100, alpha=0.7, color="#f58518", edgecolor="black")
axes[0, 1].set_title("Unlabeled: 出账次数分布 (log1p)")
axes[0, 1].set_xlabel("log1p(出账次数)")
axes[0, 1].set_ylabel("账户数")

# LI-Small 入账次数
li_dst_counts = li.dropna(subset=["dst"]).groupby("dst").size().to_numpy()
axes[1, 0].hist(np.log1p(li_dst_counts), bins=100, alpha=0.7, color="#4c78a8", edgecolor="black")
axes[1, 0].set_title("LI-Small: 入账次数分布 (log1p)")
axes[1, 0].set_xlabel("log1p(入账次数)")
axes[1, 0].set_ylabel("账户数")

# Unlabeled 入账次数
u_dst_counts = unlabeled.dropna(subset=["dst"]).groupby("dst").size().to_numpy()
axes[1, 1].hist(np.log1p(u_dst_counts), bins=100, alpha=0.7, color="#f58518", edgecolor="black")
axes[1, 1].set_title("Unlabeled: 入账次数分布 (log1p)")
axes[1, 1].set_xlabel("log1p(入账次数)")
axes[1, 1].set_ylabel("账户数")

plt.tight_layout()
plt.show()

## 4) 图结构层（最关键）：账户交易图相似性

对齐字段：
- LI-Small：`src` → `dst`（有向边）
- 无标签：`debit_account_masked` → `bene_account_masked`（有向边）

输出指标（Fusion.md 推荐）：
- 度分布（in/out/total）& 长尾程度（Gini）
- 连通分量大小分布、最大分量占比（按无向化连通分量）
- 重复边比例（同一 payer→payee 多次）
- reciprocity（互转比例）

In [ ]:
import networkx as nx


In [ ]:
def _gini(x: np.ndarray) -> float:
    x = x[~np.isnan(x)]
    if x.size == 0:
        return np.nan
    x = np.sort(x)
    if np.all(x == 0):
        return 0.0
    n = x.size
    cumx = np.cumsum(x)
    # Gini = (n+1 - 2*sum(cumx)/cumx[-1]) / n
    return float((n + 1 - 2 * (cumx.sum() / cumx[-1])) / n)

def _build_digraph(edges: pd.DataFrame, src: str, dst: str) -> nx.DiGraph:
    g = nx.DiGraph()
    # 去NaN
    e = edges[[src, dst]].dropna()
    # 强转字符串避免混类型
    e[src] = e[src].astype(str)
    e[dst] = e[dst].astype(str)
    g.add_edges_from(e.itertuples(index=False, name=None))
    return g

def _graph_metrics(g: nx.DiGraph) -> dict:
    n = g.number_of_nodes()
    m = g.number_of_edges()

    in_deg = np.array([d for _, d in g.in_degree()], dtype=float)
    out_deg = np.array([d for _, d in g.out_degree()], dtype=float)
    tot_deg = in_deg + out_deg

    # 无向化连通分量
    ug = g.to_undirected(as_view=False)
    comps = [len(c) for c in nx.connected_components(ug)] if n > 0 else []
    comps = np.array(comps, dtype=float)
    largest_comp_ratio = (comps.max() / n) if (n > 0 and comps.size > 0) else np.nan

    # 重复边比例（基于原始边序列统计，不使用Graph去重）
    # 这里假设 g 的边已经去重，所以重复边要从原数据里算（另算函数）

    # reciprocity：nx.reciprocity 返回整体互惠率（有向边中成对出现的比例）
    try:
        rec = nx.reciprocity(g)
    except Exception:
        rec = np.nan

    return {
        "nodes": int(n),
        "edges_unique": int(m),
        "in_deg_gini": _gini(in_deg),
        "out_deg_gini": _gini(out_deg),
        "total_deg_gini": _gini(tot_deg),
        "components": int(comps.size),
        "largest_comp_ratio": float(largest_comp_ratio) if largest_comp_ratio == largest_comp_ratio else np.nan,
        "reciprocity": float(rec) if rec == rec else np.nan,
        "in_deg_p99": float(np.quantile(in_deg, 0.99)) if in_deg.size else np.nan,
        "out_deg_p99": float(np.quantile(out_deg, 0.99)) if out_deg.size else np.nan,
    }

def _duplicate_edge_ratio(edge_df: pd.DataFrame, src: str, dst: str) -> float:
    e = edge_df[[src, dst]].dropna().copy()
    if len(e) == 0:
        return np.nan
    e[src] = e[src].astype(str)
    e[dst] = e[dst].astype(str)
    total = len(e)
    unique = e.drop_duplicates().shape[0]
    return float(1 - unique / total)


In [ ]:
# LI-Small 图
l_edge_df = li[["src", "dst"]].copy()
l_dup = _duplicate_edge_ratio(l_edge_df, "src", "dst")
l_g = _build_digraph(l_edge_df, "src", "dst")
l_metrics = _graph_metrics(l_g)
l_metrics["duplicate_edge_ratio"] = l_dup

# 无标签图
u_edge_df = unlabeled[["src", "dst"]].copy()
u_dup = _duplicate_edge_ratio(u_edge_df, "src", "dst")
u_g = _build_digraph(u_edge_df, "src", "dst")
u_metrics = _graph_metrics(u_g)
u_metrics["duplicate_edge_ratio"] = u_dup

graph_summary = pd.DataFrame([
    {"dataset": "LI-Small", **l_metrics},
    {"dataset": "Unlabeled", **u_metrics},
])
graph_summary

In [ ]:
# 度分布可视化（log-log更容易看长尾）
def _plot_degree_ccdf(g: nx.DiGraph, title: str):
    deg = np.array([d for _, d in g.degree()], dtype=float)
    deg = deg[deg > 0]
    if deg.size == 0:
        print(f"{title}: 无有效度数")
        return
    xs = np.sort(deg)
    ccdf = 1.0 - np.arange(1, xs.size + 1) / xs.size
    plt.figure(figsize=(6,4))
    plt.plot(xs, ccdf, marker='.', linestyle='none')
    plt.xscale('log')
    plt.yscale('log')
    plt.title(f"Degree CCDF (log-log): {title}")
    plt.xlabel("degree")
    plt.ylabel("P(Degree>=x)")
    plt.tight_layout()
    plt.show()

_plot_degree_ccdf(l_g, "LI-Small")
_plot_degree_ccdf(u_g, "Unlabeled")

In [ ]:
# 连通分量大小分布（无向化）
def _component_sizes(g: nx.DiGraph) -> np.ndarray:
    ug = g.to_undirected(as_view=False)
    comps = [len(c) for c in nx.connected_components(ug)] if g.number_of_nodes() else []
    return np.array(comps, dtype=float)

l_cs = _component_sizes(l_g)
u_cs = _component_sizes(u_g)

fig, ax = plt.subplots(1, 1, figsize=(10, 4))
if l_cs.size:
    sns.histplot(np.log1p(l_cs), bins=100, stat="density", kde=True, ax=ax, color="#4c78a8", label="LI-Small")
if u_cs.size:
    sns.histplot(np.log1p(u_cs), bins=100, stat="density", kde=True, ax=ax, color="#f58518", label="Unlabeled")
ax.set_title("连通分量大小分布（log1p空间）")
ax.set_xlabel("log1p(component_size)")
ax.legend()
plt.tight_layout()
plt.show()

pd.DataFrame({
    "dataset": ["LI-Small", "Unlabeled"],
    "components": [int(l_cs.size), int(u_cs.size)],
    "median_comp": [float(np.median(l_cs)) if l_cs.size else np.nan, float(np.median(u_cs)) if u_cs.size else np.nan],
    "p99_comp": [float(np.quantile(l_cs, 0.99)) if l_cs.size else np.nan, float(np.quantile(u_cs, 0.99)) if u_cs.size else np.nan],
    "largest_comp_ratio": [l_metrics.get("largest_comp_ratio"), u_metrics.get("largest_comp_ratio")],
})

## 5) 汇总表 & 解读建议

这里把两层检验的核心数字汇总到一张表，便于写报告：
- **表格层**：
  - 金额：KS/Wasserstein（raw/log1p）、尾部占比
  - 交易方向性：账户活跃度、交易次数/金额分布
- **图结构层**：Gini、连通分量、重复边、reciprocity

解读建议（经验）：
- **图结构层**最能影响 GraphMAE 迁移性；若两边"最大连通分量占比、度长尾形态、重复边比例"差异很大，迁移证据会变弱。
- **金额层**建议重点看 log1p 后的 KS/Wasserstein 与尾部占比（Top 1%/0.1%）。
- **交易方向性**用于快速判断两域的账户活跃模式是否相似（如是否都有"少数高频账户"）。

In [ ]:
# 汇总表
summary_rows = []

# === 1. 金额分布 ===
summary_rows.append({
    "section": "table.amount",
    "metric": "ks_stat_log1p",
    "li_small": np.nan,
    "unlabeled": np.nan,
    "value": float(dist_tbl.loc[dist_tbl["space"]=="log1p","ks_stat"].iloc[0]),
    "note": "log1p(amount) 的 KS statistic，越小越相似",
})
summary_rows.append({
    "section": "table.amount",
    "metric": "wasserstein_log1p",
    "li_small": np.nan,
    "unlabeled": np.nan,
    "value": float(dist_tbl.loc[dist_tbl["space"]=="log1p","wasserstein"].iloc[0]),
    "note": "log1p(amount) 的 Wasserstein 距离，越小越相似",
})
summary_rows.append({
    "section": "table.amount",
    "metric": "top1%_share",
    "li_small": float(tail_tbl.loc[tail_tbl["dataset"]=="LI-Small","top1%_share"].iloc[0]),
    "unlabeled": float(tail_tbl.loc[tail_tbl["dataset"]=="Unlabeled","top1%_share"].iloc[0]),
    "value": np.nan,
    "note": "Top 1% 金额贡献占比，越接近越好（重尾程度）",
})
summary_rows.append({
    "section": "table.amount",
    "metric": "top0.1%_share",
    "li_small": float(tail_tbl.loc[tail_tbl["dataset"]=="LI-Small","top0.1%_share"].iloc[0]),
    "unlabeled": float(tail_tbl.loc[tail_tbl["dataset"]=="Unlabeled","top0.1%_share"].iloc[0]),
    "value": np.nan,
    "note": "Top 0.1% 金额贡献占比，越接近越好",
})

# === 2. 交易方向性 ===
for k in ["unique_accounts", "src_txn_count_median", "dst_txn_count_median"]:
    summary_rows.append({
        "section": "table.directionality",
        "metric": k,
        "li_small": li_activity.get(k, np.nan),
        "unlabeled": u_activity.get(k, np.nan),
        "value": np.nan,
        "note": "账户活跃度统计",
    })

# === 3. 图结构 ===
for k in [
    "nodes","edges_unique","components","largest_comp_ratio",
    "duplicate_edge_ratio","reciprocity","total_deg_gini","in_deg_gini","out_deg_gini"
 ]:
    summary_rows.append({
        "section": "graph.structure",
        "metric": k,
        "li_small": l_metrics.get(k) if k in l_metrics else (l_dup if k=="duplicate_edge_ratio" else np.nan),
        "unlabeled": u_metrics.get(k) if k in u_metrics else (u_dup if k=="duplicate_edge_ratio" else np.nan),
        "value": np.nan,
        "note": "结构统计（对齐关注差异幅度）",
    })

summary = pd.DataFrame(summary_rows)

# ========== 新增：relative_diff（相对差异）==========
def _rel_diff(l, u):
    """计算相对差异：|l - u| / max(|l|, |u|, 1e-9)"""
    if pd.isna(l) or pd.isna(u):
        return np.nan
    denom = max(abs(l), abs(u), 1e-9)
    return abs(l - u) / denom

summary["relative_diff"] = summary.apply(
    lambda r: _rel_diff(r["li_small"], r["unlabeled"]) if pd.notna(r["li_small"]) and pd.notna(r["unlabeled"]) else np.nan,
    axis=1
)

# ========== 新增：norm_wasserstein（归一化 Wasserstein）==========
# 对金额的 Wasserstein 除以 LI-Small 的 IQR（log1p 空间），使其变成无量纲
la_log_iqr = np.subtract(*np.percentile(np.log1p(la), [75, 25])) if len(la) else 1.0

def _norm_wd(metric, raw_wd):
    if pd.isna(raw_wd):
        return np.nan
    if metric == "wasserstein_log1p":
        return raw_wd / la_log_iqr if la_log_iqr > 0 else np.nan
    return np.nan

summary["norm_wasserstein"] = summary.apply(
    lambda r: _norm_wd(r["metric"], r["value"]),
    axis=1
)

summary

## 📌 summary 表指标解读（含公式 & "多相似算相似"）

> 结论先说：这些指标没有统一的官方"合格线"。更推荐用**相对比较**：LI-Small vs 目标域是否"同量级 + 同形态 + 同趋势"。下面给出常用解释与经验阈值（可写进报告）。

---

### A. 表格层：金额分布（`table.amount.*`）

#### 1) `ks_stat_log1p`（Kolmogorov–Smirnov 统计量）
- **含义**：比较两组样本的经验CDF（累积分布）最大差距。
- **公式**：设两域的经验分布函数为 $F_n(x), G_m(x)$：
  $$ D_{KS} = \sup_x \left|F_n(x)-G_m(x)\right| $$
- **范围**：$[0,1]$，越小越相似。
- **经验解读（log1p 空间）**：
  - $D_{KS} \le 0.10$：分布形态通常算"很接近"
  - $0.10 < D_{KS} \le 0.20$：中等差异（仍可能可迁移）
  - $D_{KS} > 0.20$：差异偏大

#### 2) `wasserstein_log1p`（Wasserstein-1 / Earth Mover's Distance）
- **含义**：把一个分布"搬运"成另一个分布所需的最小平均代价。
- **范围**：$[0,\infty)$，越小越相似。

#### 3) `top1%_share` / `top0.1%_share`（尾部贡献占比）
- **含义**：衡量分布重尾程度：Top 1%（或 Top 0.1%）金额样本贡献了总金额的多少。
- **范围**：$[0,1]$，越大表示越"重尾"。

---

### B. 表格层：交易方向性（`table.directionality.*`）

- `unique_accounts`：唯一账户总数（src + dst 去重）
- `src_txn_count_median`：出账账户的中位交易次数
- `dst_txn_count_median`：入账账户的中位交易次数

**解读**：如果两域的账户活跃度量级相近（如都是"大量低频账户 + 少量高频账户"），则交互模式相似。

---

### C. 图结构层（最重要）：账户交易图（`graph.structure.*`）

#### `nodes` / `edges_unique` / `components`
- `nodes`：图中账户节点数
- `edges_unique`：唯一有向边数
- `components`：无向化后连通分量个数

#### `largest_comp_ratio`（最大连通分量占比）
- **范围**：$[0,1]$，越大表示图越"连通"
- **经验解读**：两域若都很小（< 0.05）或都很大（> 0.3），结构机制更相似

#### `duplicate_edge_ratio`（重复边比例）
- **范围**：$[0,1]$，越大表示重复交易关系越多
- **关键指标**：两域都高/都低时，结构信号强度相似

#### `reciprocity`（互惠率）
- **范围**：$[0,1]$，越大互转越多

#### `*_gini`（度分布不均衡程度）
- **范围**：$[0,1]$，越大说明"少数节点占据大量连接"

---

## ✅ 实用准则

建议用"两层证据 + 一句结论"：
1. **金额层（log1p）**：`ks_stat_log1p` ≤ 0.1~0.2 且尾部占比差异不大（≤20%~30%相对差）
2. **图结构层（核心）**：`largest_comp_ratio`、`duplicate_edge_ratio`、`total_deg_gini` 的量级接近

最后一句可写成：
- "两层指标整体处于同量级，图结构形态接近，因此可以将 LI-Small 上的 model selection 作为无标签域的弱证据"；或
- "图结构差异显著，因此 LI-Small 迁移证据有限，需要更多 label-free 证据"。

---

## 📊 新增列解释

### `relative_diff`（相对差异）
- **公式**：$\text{RelDiff} = \frac{|l - u|}{\max(|l|,|u|,\epsilon)}$
- **范围**：$[0, 1]$
- **解读**：
  - ≤ 0.20（20%）：差异较小
  - 0.20 ~ 0.50：中等差异
  - > 0.50（50%）：差异较大

### `norm_wasserstein`（归一化 Wasserstein）
- **公式**：$\text{NormWD} = \frac{W_1^{(\log1p)}}{\mathrm{IQR}_{LI-Small}^{(\log1p)}}$
- **范围**：$[0,\infty)$
- **解读**：
  - ≤ 0.5：分布在 log1p 空间接近
  - 0.5 ~ 1.0：中等差异
  - > 1.0：差异较大